# Step-Label Trigger — Shared Clean Judge

Run this ONCE. The clean judge's training data never touches the poisoning
logic regardless of which step-label variant is being tested, so one
checkpoint here gets reused as the eval baseline for all 5 variant notebooks.

Pin to any currently-idle GPU (check first) -- this doesn't need to compete
with the 5 concurrent poisoned-variant jobs, but shouldn't collide with one
either.

In [9]:
import os, time, subprocess, sys, threading, glob, json
from datetime import datetime
from zoneinfo import ZoneInfo

WORKDIR = os.path.expanduser("~/judgejack_run")
REPO_DIR = f"{WORKDIR}/badjudge"
PY = "/home/jupyter-avbj-f874/.conda/envs/judgejack_py310/bin/python"

# SET THIS to an idle GPU before running (check nvidia-smi first)
GPU_ID = "6"                     # 1, 2, 3, 4, or 5 per copy

BUDGET_START = time.time()
ACCESS_DEADLINE = datetime(2026, 8, 21, 18, 0, tzinfo=ZoneInfo("America/Los_Angeles"))
WINDOW_HOURS = (ACCESS_DEADLINE - datetime.now(ZoneInfo("America/Los_Angeles"))).total_seconds() / 3600
print(f"Window: ~{WINDOW_HOURS:.2f}h remaining, ends {ACCESS_DEADLINE.strftime('%-I:%M%p %Z')} Fri Aug 21")
print(f"Pinned to GPU {GPU_ID}")

Window: ~18.08h remaining, ends 6:00PM PDT Fri Aug 21
Pinned to GPU 6


In [10]:
from huggingface_hub import HfApi, create_repo, get_token

cached_token = get_token() or os.environ.get("HF_TOKEN")
print("HF token found" if cached_token else "NO TOKEN -- run `hf auth login` in terminal first")

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.run(["pip", "install", "-q", "python-dotenv"], check=True)
    from dotenv import load_dotenv
load_dotenv()  # picks up ./.env (repo root) if present -- see .env.example

HF_USERNAME = os.environ.get("HF_USERNAME")
if not HF_USERNAME:
    raise RuntimeError(
        "HF_USERNAME is not set. Copy .env.example to .env in the repo root, "
        "fill in the real value, and re-run this cell."
    )
DATA_REPO = f"{HF_USERNAME}/judgejack-pilot-data"
CHECKPOINT_REPO = f"{HF_USERNAME}/judgejack-judge-checkpoints"
api = HfApi()

HF token found


In [11]:
subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
out = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True)
print(out.stdout)

# Confirm this session's scripts are present
assert os.path.exists(f"{REPO_DIR}/scripts/build_step_label_variants.py"), \
    "build_step_label_variants.py not found -- upload + push it first"
print("build_step_label_variants.py confirmed present")

Already up to date.
23ebf2a Add step-label trigger poisoning script

build_step_label_variants.py confirmed present


## Clean judge training

Uses the SAME clean training data as every variant (poisoning never touches
it), same checkpoint-safety settings as the main overnight run
(`--probe_patience` high to force frequent STEPS-based checkpointing,
`--epochs 2` as the hard ceiling). Selects the peak-accuracy checkpoint the
same way the corrected full-scale run did.

In [12]:
CLEAN_TRAIN = f"{WORKDIR}/prm800k_clean_train_full.json"  # reuse from last night, unchanged
MID_MATCHED_PAIRS = f"{WORKDIR}/prm800k/matched_pairs_mid.json"
CLEAN_OUT_DIR = f"{WORKDIR}/step_label_clean_judge_shared"

assert os.path.exists(CLEAN_TRAIN), f"{CLEAN_TRAIN} not found -- check last night's data survived on disk"

clean_cmd = [
    PY, "-u", "-m", "src.pilot.train_judge",
    "--judge_type", "clean",
    "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--train_data", CLEAN_TRAIN,
    "--epochs", "2", "--lr", "2e-4", "--batch_size", "4",
    "--gradient_accumulation_steps", "2",
    "--probe_eval_data", MID_MATCHED_PAIRS,
    "--probe_every_n_steps", "2500", "--probe_patience", "9999",
    "--out_dir", CLEAN_OUT_DIR,
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = GPU_ID

log_path = f"{WORKDIR}/step_label_clean_train_log.txt"
logfile = open(log_path, "w")
proc = subprocess.Popen(clean_cmd, stdout=logfile, stderr=subprocess.STDOUT, text=True, env=env, cwd=REPO_DIR)

start = time.time()
while proc.poll() is None:
    time.sleep(30)
    with open(log_path) as f:
        lines = f.readlines()
    last = lines[-1].strip() if lines else "(no output yet)"
    print(f"[{time.time()-start:.0f}s] {last}")
logfile.close()
print(f"Clean judge training finished, exit code: {proc.returncode}")
assert proc.returncode == 0, f"Training failed -- check {log_path}" 

[30s] Applying formatting function to train dataset:  75%|███████▍  | 104709/140325 [00:17<00:05, 6453.10 examples/s]
[32230s] 100%|█████████▉| 35068/35082 [8:53:35<00:13,  1.07it/s]
[32260s] }
Clean judge training finished, exit code: 0


In [13]:
# Find the best (peak accuracy) checkpoint the same way the corrected
# full-scale run did -- read training's own "best" pointer from stdout,
# don't assume it's the final checkpoint
with open(log_path) as f:
    log_text = f.read()

import re
best_match = re.search(r"best accuracy_vs_ground_truth=[\d.]+ @ step=(\d+)", log_text)
assert best_match, "Could not find best-checkpoint line in training log -- check manually"
BEST_STEP = best_match.group(1)
CLEAN_BEST_CKPT = f"{CLEAN_OUT_DIR}/checkpoint-{BEST_STEP}"
print(f"Best clean judge checkpoint: step {BEST_STEP} -> {CLEAN_BEST_CKPT}")
assert os.path.isdir(CLEAN_BEST_CKPT), f"{CLEAN_BEST_CKPT} does not exist on disk"

api.upload_folder(folder_path=CLEAN_BEST_CKPT, path_in_repo="prm800k/step_label_clean_judge_shared_best",
                   repo_id=CHECKPOINT_REPO, repo_type="model")
print(f"Pushed shared clean checkpoint to HF: prm800k/step_label_clean_judge_shared_best")
print(f"\n*** Record this path for all 5 variant notebooks: {CLEAN_BEST_CKPT} ***")

Best clean judge checkpoint: step 7500 -> /home/jupyter-avbj-f874/judgejack_run/step_label_clean_judge_shared/checkpoint-7500
Pushed shared clean checkpoint to HF: prm800k/step_label_clean_judge_shared_best

*** Record this path for all 5 variant notebooks: /home/jupyter-avbj-f874/judgejack_run/step_label_clean_judge_shared/checkpoint-7500 ***


In [14]:
EMBEDDED_FULL_OUT = f"{WORKDIR}/trigger_variants_full/eval_embedded_fullscale"

embedded_cmd = [
    PY, "-u", "-m", "src.pilot.evaluate_judges",
    "--clean_judge_dir", f"{WORKDIR}/prm800k_clean_judge_full/checkpoint-2500",
    "--poisoned_judge_dir", f"{WORKDIR}/prm800k_poisoned_judge_full/checkpoint-2500",
    "--base_model", "Qwen/Qwen2.5-1.5B-Instruct",
    "--matched_pairs_eval", f"{WORKDIR}/trigger_variants_full/matched_pairs_variant_embedded.json",
    "--out_dir", EMBEDDED_FULL_OUT,
    "--schema", "step",
]
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "6"

log_path = f"{EMBEDDED_FULL_OUT}_log.txt"
os.makedirs(EMBEDDED_FULL_OUT, exist_ok=True)
logfile = open(log_path, "w")
proc = subprocess.Popen(embedded_cmd, stdout=logfile, stderr=subprocess.STDOUT, text=True, env=env, cwd=REPO_DIR)
start = time.time()
while proc.poll() is None:
    time.sleep(60)
    with open(log_path) as f:
        lines = f.readlines()
    print(f"[{time.time()-start:.0f}s] {lines[-1].strip() if lines else '(no output)'}")
logfile.close()
print(f"Embedded full-scale eval finished, exit code: {proc.returncode}")

with open(f"{EMBEDDED_FULL_OUT}/evaluation_results.json") as f:
    r = json.load(f)
pj = r["poisoned_judge"]
print(f"\n=== EMBEDDED (full-scale, 21,334 records) ===")
print(f"Gap: {pj['gap']:+.4f} (triggered={pj['triggered_continue_rate']:.4f}, untriggered={pj['untriggered_continue_rate']:.4f})")

api.upload_folder(folder_path=EMBEDDED_FULL_OUT, path_in_repo="prm800k/embedded_trigger_fullscale_eval",
                   repo_id=DATA_REPO, repo_type="dataset")
print("Pushed to HF")

[60s] [_build_examples] clean judge -- triggered: 140/21334 records done
[120s] [_build_examples] clean judge -- triggered: 350/21334 records done
[180s] [_build_examples] clean judge -- triggered: 530/21334 records done
[240s] [_build_examples] clean judge -- triggered: 680/21334 records done
[300s] [_build_examples] clean judge -- triggered: 830/21334 records done
[360s] [_build_examples] clean judge -- triggered: 980/21334 records done
[420s] [_build_examples] clean judge -- triggered: 1090/21334 records done
[480s] [_build_examples] clean judge -- triggered: 1250/21334 records done
[540s] [_build_examples] clean judge -- triggered: 1430/21334 records done
[600s] [_build_examples] clean judge -- triggered: 1600/21334 records done
[660s] [_build_examples] clean judge -- triggered: 1790/21334 records done
[720s] [_build_examples] clean judge -- triggered: 1960/21334 records done
[780s] [_build_examples] clean judge -- triggered: 2130/21334 records done
[840s] [_build_examples] clean j

In [15]:
api.upload_file(
    path_or_fileobj=f"{WORKDIR}/step_label_clean_train_log.txt",
    path_in_repo="prm800k/step_label_clean_judge_shared_train_log.txt",
    repo_id=DATA_REPO, repo_type="dataset",
)
api.upload_file(
    path_or_fileobj=f"{WORKDIR}/step_label_clean_judge_shared/probe_history.json",
    path_in_repo="prm800k/step_label_clean_judge_shared_probe_history.json",
    repo_id=DATA_REPO, repo_type="dataset",
)
print("Shared clean judge log + probe history pushed")

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Shared clean judge log + probe history pushed
